# pcm: ancestral state reconstruction

This notebook demonstrates fitting discrete-state continuous-time Markov models (CTMC) and inferring ancestral states with `infer_ancestral_states_discrete_ctmc()`. This example fits a discrete-state continuous-time Markov model (Mk) on a phylogeny, estimates transition rates by maximum likelihood, and then computes marginal ancestral-state posteriors conditional on the fitted parameters.

In [35]:
import toytree

In [36]:
# example tree: simulate a birth-death tree with 25 tips
tree = toytree.rtree.bdtree(25, seed=123)

### Simulate tip and ancestral states
See [link] for more details on simulating discrete characters on trees. In this case we simulate data only across tips a 3-state character across the tree. Take note of the visualization of these data on the tree below. Here we simulated the data under an equal-rates model (ER), and the true ancestral root state is "C" (blue). We will assess how well we can reconstruct this truth based on fitting models to an incomplete observation of the data: trait data for only the tips of the phylogeny. 

In [41]:
# simulate trait 'X' and store to tree data
tree.pcm.simulate_discrete_trait(
    nstates=3,
    model="ER",
    inplace=True,
    seed=12345,
    trait_name="X",
    state_names="ABC",
)

In [42]:
# examine trait 'X' in tree data
tree.get_node_data().head()

,idx,name,height,dist,support,X
0,0,r0,4.440892e-16,2.219243,NaN,C
1,1,r1,0.000000e+00,0.813243,NaN,C
2,2,r2,0.000000e+00,0.125646,NaN,A
3,3,r3,0.000000e+00,0.125646,NaN,A
4,4,r4,0.000000e+00,0.617988,NaN,A


In [43]:
# visualize a discrete 3-state trait 'X' on the tree
tree.draw(node_sizes=8, node_colors=("X", "Set2"), node_mask=False, layout="d");

<svg class="toyplot-canvas-Canvas" xmlns:toyplot="http://www.sandia.gov/toyplot" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="513.9999999999998px" height="300.0px" viewBox="0 0 513.9999999999998 300.0" preserveAspectRatio="xMidYMid meet" style="background-color:transparent;border-color:#292724;border-style:none;border-width:1.0;fill:rgb(16.1%,15.3%,14.1%);fill-opacity:1.0;font-family:Helvetica;font-size:12px;opacity:1.0;stroke:rgb(16.1%,15.3%,14.1%);stroke-opacity:1.0;stroke-width:1.0" id="t61fa1a26a22b42388470b818f5acf824"> r0 r1 r2 r3 r4 r5 r6 r7 r8 r9 r10 r11 r12 r13 r14 r15 r16 r17 r18 r19 r20 r21 r22 r23 r24

## Fit a discrete CTMC model 
To infer ancestral states of a discrete character we will fit a continuous time markov chain (CTMC) model. The function `fit_discrete_ctmc` can be used to fit this model under a range of parameters that allow for defining nested models with different constraints or free parameters, defining the complexity of the model. 

### ER
The simplest is "equal-rates" (ER), where there is only one rate parameter for transitions between all character states. 

### SYM
...

### ARD
...

In [44]:
# return a fitted model object
tree.pcm.fit_discrete_ctmc("X", nstates=3, model="ER")

PCMDiscreteCTMCFitResult(
  model=ER, nstates=3, nparams=1
  log_likelihood=-24.6709
  state_frequencies=[0.3333, 0.3333, 0.3333]
  relative_rates=<array 3x3>
  qmatrix=<array 3x3>
  fixed_rates=<array 3x3>
  fixed_state_frequencies=None
)

### Simulate only tip states
This is typically the case for empirical data. We observe data at the tips and want to infer the ancestral states under a model of evolution. Here, the data are generated under an equal-rates 3-state model and stored directly to the tree as a trait named "X".

In [17]:
# simulate discrete 3-state trait under ER model, keeping only tip states
tree.pcm.simulate_discrete_trait(
    nstates=3,
    model="ER",
    inplace=True,
    trait_name="X",
    state_names="ABC",
    tips_only=True,
    seed=12345,
)

# visualize trait with unknown internal values
tree.draw(node_sizes=8, node_colors=("X", "Set2"), node_mask=False, layout="d");

<svg class="toyplot-canvas-Canvas" xmlns:toyplot="http://www.sandia.gov/toyplot" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="513.9999999999998px" height="300.0px" viewBox="0 0 513.9999999999998 300.0" preserveAspectRatio="xMidYMid meet" style="background-color:transparent;border-color:#292724;border-style:none;border-width:1.0;fill:rgb(16.1%,15.3%,14.1%);fill-opacity:1.0;font-family:Helvetica;font-size:12px;opacity:1.0;stroke:rgb(16.1%,15.3%,14.1%);stroke-opacity:1.0;stroke-width:1.0" id="t802a836a6d0b460eae2237e8106e1d34"> r0 r1 r2 r3 r4 r5 r6 r7 r8 r9 r10 r11 r12 r13 r14 r15 r16 r17 r18 r19 r20 r21 r22 r23 r24

## Infer ancestral states
The method `infer_ancestral_states_discrete_ctmc` from the `pcm` subpackage can be used to fit a CTMC model and infer ancestral character states at internal nodes of the tree as posterior probabilities. Under the hood it fits a CTMC and returns a `PCMDiscreteCTMCFitResult` object just like above. However, in addition, using this fit model it infers the posterior probability of each state across each internal node of the tree and returns a DataFrame with both these posteriors and the most likely state at each node. These results are returned in a dict along with the original tree with the ancestral state results mapped as data to nodes. 

In [18]:
# fit a CTMC model and ancestral state reconstruction
result = tree.pcm.infer_ancestral_states_discrete_ctmc(
    "X", nstates=3, model="ER", inplace=True
)

In [19]:
# the fitted model object with details of the Q rate matrix
result["model_fit"]

PCMDiscreteCTMCFitResult(
  model=ER, nstates=3, nparams=1
  log_likelihood=-18.1497
  state_frequencies=[0.3333, 0.3333, 0.3333]
  relative_rates=<array 3x3>
  qmatrix=<array 3x3>
  fixed_rates=<array 3x3>
  fixed_state_frequencies=None
)

In [20]:
# the inferred state and posterior prob at each node
result["data"].head()

,X_anc,X_anc_posterior
0,C,"(0.0, 0.0, 1.0)"
1,C,"(0.0, 0.0, 1.0)"
2,A,"(1.0, 0.0, 0.0)"
3,A,"(1.0, 0.0, 0.0)"
4,A,"(1.0, 0.0, 0.0)"


In [23]:
# draw inferred character states on tree
c, a, m = tree.draw(layout="r", tip_labels=False, height=375)
tree.set_node_data_from_dataframe(result["data"], inplace=True)
tree.annotate.add_edges(a, color=("X_anc", "Set2"), width=4, use_color_gradient=True)
tree.annotate.add_node_pie_markers(
    a, "X_anc_posterior", colors="Set2", size=10, istroke_width=1
);

<svg class="toyplot-canvas-Canvas" xmlns:toyplot="http://www.sandia.gov/toyplot" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="300.0px" height="375.0px" viewBox="0 0 300.0 375.0" preserveAspectRatio="xMidYMid meet" style="background-color:transparent;border-color:#292724;border-style:none;border-width:1.0;fill:rgb(16.1%,15.3%,14.1%);fill-opacity:1.0;font-family:Helvetica;font-size:12px;opacity:1.0;stroke:rgb(16.1%,15.3%,14.1%);stroke-opacity:1.0;stroke-width:1.0" id="td6e821c27c244f62b02eed11e37d7ae3"> 0, 0, 1 0, 0, 1 1, 0, 0 1, 0, 0 1, 0, 0 1, 0, 0 1, 0, 0 0, 1, 0 0, 1, 0 1, 0, 0 1, 0, 0 0, 0, 1 0, 0, 1 0, 0, 1 0, 0, 1 0, 0, 1 0, 0, 1 0, 0, 1 0, 1, 0 0, 1, 0 0, 1, 0 0, 1, 0 0, 0, 1 0, 0, 1 0, 0, 1 0.975864, 0.024071, 6.49533e-05 0.648225, 0.101692, 0.250083 0.905115, 0.0209112, 0.0739734 0.94811, 0.0518027, 8.68969e-05 0.693129, 0.301431, 0.00543966 0.0043348, 0.994246, 0.00141892 0.214328, 0.782458, 0.0032141 0.497381, 0.462237, 0.0403824 0.630925, 0.249559, 0.119515 0.495178, 0.40133, 0.103492 8.35744e-05, 9.45851e-07, 0.999915 4.73046e-05, 7.73584e-07, 0.999952 0.000323309, 3.62864e-07, 0.999676 0.0681581, 0.00353643, 0.928305 1.92385e-05, 0.999965, 1.54218e-05 1.89766e-05, 0.99927, 0.000710711 4.76638e-05, 0.998377, 0.00157573 0, 0, 1 0.0165048, 0.000644307, 0.982851 0.0612058, 0.0613279, 0.877466 0.293392, 0.0265844, 0.680023 0.432021, 0.0425045, 0.525475 0.47364, 0.0499979, 0.476362 0.999995, 1.62549e-08, 4.71181e-06

### Infer ancestral states under ARD model
Under this more complex model we can see that the ancestral state reconstruction looks quite different. Here state A (green) is strongly inferred as the ancestral root state. The inferred transition rates in the CTMC model can greatly affect the results. Beware, however, that on relatively small trees like this one, where there are few transitions, a complex model like the all-rates-different (ARD) model may infer highly unequal transition rates based on few observations. See below for how to use Model Selection to choose the model that is most appropriate for your data.

In [24]:
result = tree.pcm.infer_ancestral_states_discrete_ctmc(
    "X", nstates=3, model="ARD", inplace=True
)

# draw inferred character states on tree
c, a, m = tree.draw(layout="r", tip_labels=False, height=375)
tree.annotate.add_edges(a, color=("X_anc", "Set2"), width=4, use_color_gradient=True)
tree.annotate.add_node_pie_markers(
    a, "X_anc_posterior", colors="Set2", size=10, istroke_width=1
);

<svg class="toyplot-canvas-Canvas" xmlns:toyplot="http://www.sandia.gov/toyplot" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="300.0px" height="375.0px" viewBox="0 0 300.0 375.0" preserveAspectRatio="xMidYMid meet" style="background-color:transparent;border-color:#292724;border-style:none;border-width:1.0;fill:rgb(16.1%,15.3%,14.1%);fill-opacity:1.0;font-family:Helvetica;font-size:12px;opacity:1.0;stroke:rgb(16.1%,15.3%,14.1%);stroke-opacity:1.0;stroke-width:1.0" id="tf81aff9a095d441fa78b136d959984a0"> 0, 0, 1 0, 0, 1 1, 0, 0 1, 0, 0 1, 0, 0 1, 0, 0 1, 0, 0 0, 1, 0 0, 1, 0 1, 0, 0 1, 0, 0 0, 0, 1 0, 0, 1 0, 0, 1 0, 0, 1 0, 0, 1 0, 0, 1 0, 0, 1 0, 1, 0 0, 1, 0 0, 1, 0 0, 1, 0 0, 0, 1 0, 0, 1 0, 0, 1 0.975864, 0.024071, 6.49533e-05 0.648225, 0.101692, 0.250083 0.905115, 0.0209112, 0.0739734 0.94811, 0.0518027, 8.68969e-05 0.693129, 0.301431, 0.00543966 0.0043348, 0.994246, 0.00141892 0.214328, 0.782458, 0.0032141 0.497381, 0.462237, 0.0403824 0.630925, 0.249559, 0.119515 0.495178, 0.40133, 0.103492 8.35744e-05, 9.45851e-07, 0.999915 4.73046e-05, 7.73584e-07, 0.999952 0.000323309, 3.62864e-07, 0.999676 0.0681581, 0.00353643, 0.928305 1.92385e-05, 0.999965, 1.54218e-05 1.89766e-05, 0.99927, 0.000710711 4.76638e-05, 0.998377, 0.00157573 0, 0, 1 0.0165048, 0.000644307, 0.982851 0.0612058, 0.0613279, 0.877466 0.293392, 0.0265844, 0.680023 0.432021, 0.0425045, 0.525475 0.47364, 0.0499979, 0.476362 0.999995, 1.62549e-08, 4.71181e-06

## Model selection
You can fit many different nested models and compare their likelihoods under a model selection criterion (AIC) that penalizes more complex models. Here we can see that the "ER" model has 92% of the AIC weight, suggesting it is the best model to use for this dataset.

In [53]:
# fit model under each of the three models
fits = {}
for model in ["ER", "SYM", "ARD"]:
    fits[model] = tree.pcm.fit_discrete_ctmc("X", nstates=3, model=model)

# get model comparison AIC table
toytree.pcm.aic_table(list(fits.values()))

,model,loglik,k,AIC,dAIC,weight_AIC,evidence_ratio_vs_best_AIC,weight,evidence_ratio_vs_best,rank_by
0,ER,-18.149664,1,38.299329,0.000000,0.920788,1.000000,0.920788,1.000000,AIC
1,SYM,-16.671723,5,43.343445,5.044117,0.073934,12.454205,0.073934,12.454205,AIC
2,ARD,-16.311314,8,48.622628,10.323299,0.005278,174.452006,0.005278,174.452006,AIC


## Fixed (fossil) internal states
Fixed states can be set on internal nodes by simply setting a value to an internal node (i.e., not 'nan'). In this case, we set an internal node to state "B" (orange) and fit the ER model as before. You can see that this greatly changes the root state, giving it a much higher probability of being in state B.

In [31]:
# get a copy of trait 'X' and fix internal node (idx=45) state to 'B'
trait = tree.get_node_data("X")
trait.loc[45] = "B"

# store this data as new trait 'Y' and store a trait 'ylabel' as well that we will use to label it
tree.set_node_data("Y", trait, inplace=True)
tree.set_node_data("ylabel", {45: "FIXED"}, inplace=True)

# infer ancestral states conditional on data "Y" (has one fixed internal state)
tree.pcm.infer_ancestral_states_discrete_ctmc("Y", nstates=3, model="ER", inplace=True)

# plot tree with ancestral states and label next to fixed state node
c, a, m = tree.draw(
    node_sizes=8, node_colors=("Y", "Set2"), node_mask=False, layout="d"
)
tree.annotate.add_node_pie_markers(
    a, "Y_anc_posterior", colors="Set2", size=10, istroke_width=1
)
tree.annotate.add_node_labels(
    a, "ylabel", mask=tree.get_node_mask(45), xshift=25, yshift=-10
);

<svg class="toyplot-canvas-Canvas" xmlns:toyplot="http://www.sandia.gov/toyplot" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="513.9999999999998px" height="300.0px" viewBox="0 0 513.9999999999998 300.0" preserveAspectRatio="xMidYMid meet" style="background-color:transparent;border-color:#292724;border-style:none;border-width:1.0;fill:rgb(16.1%,15.3%,14.1%);fill-opacity:1.0;font-family:Helvetica;font-size:12px;opacity:1.0;stroke:rgb(16.1%,15.3%,14.1%);stroke-opacity:1.0;stroke-width:1.0" id="tef3a90f19eb748d5b6aeaf07463031eb"> r0 r1 r2 r3 r4 r5 r6 r7 r8 r9 r10 r11 r12 r13 r14 r15 r16 r17 r18 r19 r20 r21 r22 r23 r24 0, 0, 1 0, 0, 1 1, 0, 0 1, 0, 0 1, 0, 0 1, 0, 0 1, 0, 0 0, 1, 0 0, 1, 0 1, 0, 0 1, 0, 0 0, 0, 1 0, 0, 1 0, 0, 1 0, 0, 1 0, 0, 1 0, 0, 1 0, 0, 1 0, 1, 0 0, 1, 0 0, 1, 0 0, 1, 0 0, 0, 1 0, 0, 1 0, 0, 1 0.989661, 0.00447132, 0.00586757 0.440367, 0.179224, 0.380409 0.306818, 0.377556, 0.315626 0.981986, 0.0102471, 0.00776691 0.843366, 0.102582, 0.0540527 0.0221816, 0.974882, 0.00293618 0.467356, 0.501347, 0.0312966 0.672562, 0.233245, 0.0941932 0.580066, 0.213383, 0.206551 0.600142, 0.268264, 0.131593 0.000144512, 0.000142764, 0.999713 0.000118589, 0.000117712, 0.999764 0.000351136, 0.000342492, 0.999306 0.0723619, 0.0674911, 0.860147 4.86708e-05, 0.999892, 5.95893e-05 0.00024903, 0.999053, 0.000697973 0.000449511, 0.998255, 0.00129527 0, 0, 1 0.0122493, 0.0305713, 0.957179 0.0802904, 0.346464, 0.573245 0, 1, 0 0.14988, 0.744829, 0.105291 0.18221, 0.70198, 0.11581 0.299903, 0.39894, 0.301157 FIXED

## Parameters

### inplace
If True, write inferred states and posterior tuples to the input tree as node features. If False, the input tree is not modified.

In [71]:
result = tree.pcm.infer_ancestral_states_discrete_ctmc(
    "X", nstates=3, model="ER", inplace=True
)
tree.get_node_data().head()

,idx,name,height,dist,support,X,X_anc,X_anc_posterior,Y,Y_anc,Y_anc_posterior,ylabel
0,0,r0,4.440892e-16,2.219243,NaN,C,C,"(0.0, 0.0, 1.0)",C,C,"(0.0, 0.0, 1.0)",NaN
1,1,r1,0.000000e+00,0.813243,NaN,C,C,"(0.0, 0.0, 1.0)",C,C,"(0.0, 0.0, 1.0)",NaN
2,2,r2,0.000000e+00,0.125646,NaN,A,A,"(1.0, 0.0, 0.0)",A,A,"(1.0, 0.0, 0.0)",NaN
3,3,r3,0.000000e+00,0.125646,NaN,A,A,"(1.0, 0.0, 0.0)",A,A,"(1.0, 0.0, 0.0)",NaN
4,4,r4,0.000000e+00,0.617988,NaN,A,A,"(1.0, 0.0, 0.0)",A,A,"(1.0, 0.0, 0.0)",NaN


### root_prior
Setting a root prior is different than fixing the character state of an internal node...

In [61]:
result = tree.pcm.infer_ancestral_states_discrete_ctmc(
    "X", nstates=3, model="ER", root_prior=[0.8, 0.1, 0.1]
)

# draw inferred character states on tree
c, a, m = tree.draw(layout="r", tip_labels=False, height=375)
tree.set_node_data_from_dataframe(result["data"], inplace=True)
tree.annotate.add_edges(a, color=("X_anc", "Set2"), width=4, use_color_gradient=True)
tree.annotate.add_node_pie_markers(
    a, "X_anc_posterior", colors="Set2", size=10, istroke_width=1
);

<svg class="toyplot-canvas-Canvas" xmlns:toyplot="http://www.sandia.gov/toyplot" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="300.0px" height="375.0px" viewBox="0 0 300.0 375.0" preserveAspectRatio="xMidYMid meet" style="background-color:transparent;border-color:#292724;border-style:none;border-width:1.0;fill:rgb(16.1%,15.3%,14.1%);fill-opacity:1.0;font-family:Helvetica;font-size:12px;opacity:1.0;stroke:rgb(16.1%,15.3%,14.1%);stroke-opacity:1.0;stroke-width:1.0" id="tf03e6df826764921ad60bbdd40d46ca6"> 0, 0, 1 0, 0, 1 1, 0, 0 1, 0, 0 1, 0, 0 1, 0, 0 1, 0, 0 0, 1, 0 0, 1, 0 1, 0, 0 1, 0, 0 0, 0, 1 0, 0, 1 0, 0, 1 0, 0, 1 0, 0, 1 0, 0, 1 0, 0, 1 0, 1, 0 0, 1, 0 0, 1, 0 0, 1, 0 0, 0, 1 0, 0, 1 0, 0, 1 0.994846, 0.00187603, 0.00327759 0.550372, 0.103433, 0.346195 0.713404, 0.0870278, 0.199569 0.994618, 0.00243775, 0.00294438 0.937566, 0.0248104, 0.0376237 0.0202819, 0.978042, 0.00167606 0.608327, 0.368227, 0.0234454 0.830726, 0.062466, 0.106808 0.716934, 0.0565787, 0.226488 0.756706, 0.0562192, 0.187075 4.28134e-05, 3.97935e-05, 0.999917 3.66257e-05, 3.51501e-05, 0.999928 0.000123148, 0.000103026, 0.999774 0.0461804, 0.0301608, 0.923659 1.58882e-05, 0.999963, 2.06937e-05 9.13797e-05, 0.99954, 0.000368896 0.000200903, 0.998964, 0.000835407 0, 0, 1 0.00603362, 0.0175646, 0.976402 0.0643732, 0.28586, 0.649767 0.244349, 0.10268, 0.652971 0.377362, 0.0844381, 0.5382 0.428415, 0.0821774, 0.489408 0.771007, 0.0637848, 0.165208

### fixed_rates
Optional ``(nstates, nstates)`` relative-rate matrix with numeric values for fixed entries and ``np.nan`` for free entries. Diagonal entries are ignored and set to zero. For ``ER`` and ``SYM`` models, mirrored off-diagonal entries must be specified symmetrically.

In [69]:
# relative rates matrix
rate_matrix = [
    [0, 1, 3],
    [1, 0, 2],
    [3, 2, 0],
]
result = tree.pcm.infer_ancestral_states_discrete_ctmc(
    "X", nstates=3, model="SYM", fixed_rates=rate_matrix
)

# draw inferred character states on tree
c, a, m = tree.draw(layout="r", tip_labels=False, height=375)
tree.set_node_data_from_dataframe(result["data"], inplace=True)
tree.annotate.add_edges(a, color=("X_anc", "Set2"), width=4, use_color_gradient=True)
tree.annotate.add_node_pie_markers(
    a, "X_anc_posterior", colors="Set2", size=10, istroke_width=1
);

<svg class="toyplot-canvas-Canvas" xmlns:toyplot="http://www.sandia.gov/toyplot" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="300.0px" height="375.0px" viewBox="0 0 300.0 375.0" preserveAspectRatio="xMidYMid meet" style="background-color:transparent;border-color:#292724;border-style:none;border-width:1.0;fill:rgb(16.1%,15.3%,14.1%);fill-opacity:1.0;font-family:Helvetica;font-size:12px;opacity:1.0;stroke:rgb(16.1%,15.3%,14.1%);stroke-opacity:1.0;stroke-width:1.0" id="tc2554007405a4f3a9ba650600f671ed0"> 0, 0, 1 0, 0, 1 1, 0, 0 1, 0, 0 1, 0, 0 1, 0, 0 1, 0, 0 0, 1, 0 0, 1, 0 1, 0, 0 1, 0, 0 0, 0, 1 0, 0, 1 0, 0, 1 0, 0, 1 0, 0, 1 0, 0, 1 0, 0, 1 0, 1, 0 0, 1, 0 0, 1, 0 0, 1, 0 0, 0, 1 0, 0, 1 0, 0, 1 0.974099, 0.00114549, 0.0247558 0.4263, 0.073274, 0.500426 0.347874, 0.144256, 0.50787 0.933434, 0.00346462, 0.0631013 0.692391, 0.0386121, 0.268997 0.00614215, 0.984677, 0.00918124 0.363931, 0.451753, 0.184317 0.547612, 0.089033, 0.363355 0.482853, 0.0667316, 0.450415 0.507873, 0.0824835, 0.409643 0.002981, 0.000439783, 0.996579 0.00233388, 0.000342075, 0.997324 0.00478944, 0.000821165, 0.994389 0.203662, 0.0673794, 0.728958 3.69208e-06, 0.999959, 3.75259e-05 2.86746e-05, 0.999665, 0.000305951 9.11621e-05, 0.999193, 0.000715421 0, 0, 1 0.0459933, 0.0538434, 0.900163 0.110737, 0.374112, 0.515151 0.301969, 0.152638, 0.545393 0.342915, 0.132028, 0.525057 0.352459, 0.128952, 0.518589 0.34717, 0.1453, 0.50753

## Example: Dollo's law
An example where `fixed_rates` is useful is when testing the hypothesis of Dollo's law, which states that complex traits evolve rarely, and are thus easier to lose than to gain. One way to test this is to infer the transition rates away from the complex state, but set the transition rate to the complex state to 0. Here, it can also be useful to fix the internal node states for when the state arose, for example if you know from fossil evidence that is was present in the common ancestor. 

In [20]:
# fix transition rates TO state A at 0
# set other states to 'nan' to be inferred
x = 0
y = "nan"
rate_matrix = [
    [0, y, y],
    [x, 0, y],
    [x, y, 0],
]

# fit model with and without using fixed_rates matrix
free_fit = tree.pcm.fit_discrete_ctmc("X", nstates=3, model="ARD")
dollo_fit = tree.pcm.fit_discrete_ctmc(
    "X", nstates=3, model="ARD", fixed_rates=rate_matrix
)

In [21]:
# print the inferred transition matrix
print(dollo_fit.qmatrix)

[[-0.65706586  0.37113402  0.28593185]
 [ 0.         -1.48001589  1.48001589]
 [ 0.          0.1052364  -0.1052364 ]]


In [26]:
# scaled relative rates matrix
print(dollo_fit.relative_rates / dollo_fit.relative_rates.max())

[[0.         0.07180076 0.19319512]
 [0.         0.         1.        ]
 [0.         0.02035936 0.        ]]


#### ancestral states

Under the Dollo model the posterior probability of the root state being "A" (green) is 1.0, since that is the only possible way that any descendants can have state "A" when no transitions to "A" are allowed. Similarly, any ancestors of tips in state "A" are also in state "A". The relatie transition rates from A->B, A->C, B->C, and C->B on the other hand are 

In [27]:
# infer ancestral states under Dollo model
tree.pcm.infer_ancestral_states_discrete_ctmc(
    "X", nstates=3, model="ARD", fixed_rates=rate_matrix, inplace=True
)
c1, a1, m1 = tree.draw(
    node_sizes=8, node_colors=("X", "Set2"), node_mask=False, layout="d"
)
tree.annotate.add_node_pie_markers(
    a1, "X_anc_posterior", colors="Set2", size=10, istroke_width=1
);

<svg class="toyplot-canvas-Canvas" xmlns:toyplot="http://www.sandia.gov/toyplot" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="513.9999999999998px" height="300.0px" viewBox="0 0 513.9999999999998 300.0" preserveAspectRatio="xMidYMid meet" style="background-color:transparent;border-color:#292724;border-style:none;border-width:1.0;fill:rgb(16.1%,15.3%,14.1%);fill-opacity:1.0;font-family:Helvetica;font-size:12px;opacity:1.0;stroke:rgb(16.1%,15.3%,14.1%);stroke-opacity:1.0;stroke-width:1.0" id="te7240ad421e747b48bd7f95450cc15d3"> r0 r1 r2 r3 r4 r5 r6 r7 r8 r9 r10 r11 r12 r13 r14 r15 r16 r17 r18 r19 r20 r21 r22 r23 r24 0, 0, 1 0, 0, 1 1, 0, 0 1, 0, 0 1, 0, 0 1, 0, 0 1, 0, 0 0, 1, 0 0, 1, 0 1, 0, 0 1, 0, 0 0, 0, 1 0, 0, 1 0, 0, 1 0, 0, 1 0, 0, 1 0, 0, 1 0, 0, 1 0, 1, 0 0, 1, 0 0, 1, 0 0, 1, 0 0, 0, 1 0, 0, 1 0, 0, 1 1, 0, 0 1, 0, 0 1, 0, 0 1, 0, 0 1, 0, 0 0.0383476, 0.961528, 0.000124119 1, 0, 0 1, 0, 0 1, 0, 0 1, 0, 0 5.255e-06, 0.000595431, 0.999399 2.4953e-06, 0.000396191, 0.999601 4.09829e-05, 0.00136092, 0.998598 0.0477092, 0.133392, 0.818899 1.13804e-06, 0.999993, 5.53644e-06 5.78443e-05, 0.999921, 2.1525e-05 0.000125551, 0.999773, 0.000101856 0, 0, 1 0.00301381, 0.0931653, 0.903821 0.074967, 0.669655, 0.255378 0.544762, 0.291324, 0.163914 0.865143, 0.0879757, 0.0468812 1, 0, 0 1, 0, 0

#### model fit
In this case, the Dollo model with only six free parameters is a better fit to the data than the all-rates-free model. This does not necessarily mean it is the correct model -- in fact we know it is not, since we simulated these data under "ER". Take caution when applying these methods to small datasets with few transitions. Nevertheless, it provides a clear demonstration of how fixing a few transition rates can greatly affect the result.

In [28]:
# print aic score
toytree.pcm.aic_table([free_fit, dollo_fit])

,model,loglik,k,AIC,dAIC,weight_AIC,evidence_ratio_vs_best_AIC,weight,evidence_ratio_vs_best,rank_by
0,ARD,-17.135817,6,46.271634,0.000000,0.764137,1.000000,0.764137,1.000000,AIC
1,ARD,-16.311314,8,48.622628,2.350994,0.235863,3.239753,0.235863,3.239753,AIC
